# Backend API Tester — MTS AI Hub
Все эндпоинты + замер времени ответа. Бэкенд: `http://localhost:8000`

In [62]:
import httpx, json, struct, io, time
from pathlib import Path

BASE = "http://localhost:8000"
USER_ID = "test_user"
TIMEOUT = 300  # 5 минут для тяжёлых моделей

def ok(label, resp, elapsed=None):
    icon = '✅' if resp.status_code < 400 else '❌'
    t = f' ({elapsed:.1f}s)' if elapsed else ''
    print(f"{icon} [{resp.status_code}] {label}{t}")
    try: print(json.dumps(resp.json(), ensure_ascii=False, indent=2)[:600])
    except: print(resp.text[:400])
    print()

def timed_get(url, **kw):
    t0 = time.time()
    r = httpx.get(url, timeout=TIMEOUT, **kw)
    return r, time.time() - t0

def timed_post(url, **kw):
    t0 = time.time()
    r = httpx.post(url, timeout=TIMEOUT, **kw)
    return r, time.time() - t0

# Создаём тестовые файлы
Path('/tmp/test_doc.txt').write_text('Это тестовый документ. Python — мощный язык. FastAPI быстрый.', encoding='utf-8')

from docx import Document
doc = Document()
doc.add_heading('Тестовый документ', level=1)
doc.add_paragraph('Python — мощный язык. MTS AI Hub — платформа для ИИ.')
doc.save('/tmp/test_doc.docx')

pdf = b'%PDF-1.4\n1 0 obj<</Type/Catalog/Pages 2 0 R>>endobj\n2 0 obj<</Type/Pages/Kids[3 0 R]/Count 1>>endobj\n3 0 obj<</Type/Page/Parent 2 0 R/MediaBox[0 0 612 792]/Contents 4 0 R/Resources<</Font<</F1 5 0 R>>>>>>endobj\n4 0 obj<</Length 44>>stream\nBT /F1 12 Tf 72 700 Td (Test PDF) Tj ET\nendstream\nendobj\n5 0 obj<</Type/Font/Subtype/Type1/BaseFont/Helvetica>>endobj\nxref\n0 6\n0000000000 65535 f \ntrailer<</Size 6/Root 1 0 R>>\nstartxref\n0\n%%EOF'
Path('/tmp/test_doc.pdf').write_bytes(pdf)

sr = 16000
wav_data = b'\x00\x00' * sr
wav = io.BytesIO()
wav.write(b'RIFF')
wav.write(struct.pack('<I', 36 + len(wav_data)))
wav.write(b'WAVEfmt ')
wav.write(struct.pack('<IHHIIHH', 16, 1, 1, sr, sr*2, 2, 16))
wav.write(b'data')
wav.write(struct.pack('<I', len(wav_data)))
wav.write(wav_data)
Path('/tmp/test_audio.wav').write_bytes(wav.getvalue())

print('✅ Файлы: test_doc.txt, test_doc.pdf, test_doc.docx, test_audio.wav')

✅ Файлы: test_doc.txt, test_doc.pdf, test_doc.docx, test_audio.wav


## 1. Health Check

In [63]:
r, t = timed_get(f"{BASE}/v1/health")
ok("Health Check", r, t)

✅ [200] Health Check (0.7s)
{
  "status": "ok",
  "services": {
    "postgres": true,
    "mws_api": true,
    "image_gen": "mws(qwen-image)",
    "asr": "mws(whisper-turbo-local)"
  }
}



## 2. Chat — Text (mws-gpt-alpha)

In [64]:
r, t = timed_post(f"{BASE}/v1/chat/completions", json={
    "model": "mws-gpt-alpha",
    "messages": [{"role": "user", "content": "Привет! Кто ты?"}],
    "stream": False, "user": USER_ID
})
ok("Chat / Text (mws-gpt-alpha)", r, t)

✅ [200] Chat / Text (mws-gpt-alpha) (2.9s)
{
  "id": "chatcmpl-92d7b6c3de5ac8e1",
  "created": 1775943052,
  "model": "mws-gpt-alpha",
  "object": "chat.completion",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Привет! Я - MWS GPT, большая языковая модель, основанная на архитектуре GPT. Я создана для помощи и ответов на ваши вопросы. Моя основная цель - предоставлять точную, ясную, полезную и содержательную информацию на разные темы и вопросы. Я всегда готов помочь и ответить на любой ваш вопрос!",
        "role": "assistant"
      }
    }
  ],
  "usage": {
    "completion



## 3. Chat — Code (qwen3-coder-480b-a35b)

In [65]:
r, t = timed_post(f"{BASE}/v1/chat/completions", json={
    "model": "qwen3-coder-480b-a35b",
    "messages": [{"role": "user", "content": "Напиши функцию сортировки пузырьком на Python"}],
    "stream": False, "user": USER_ID
})
ok("Chat / Code (qwen3-coder-480b-a35b)", r, t)

✅ [200] Chat / Code (qwen3-coder-480b-a35b) (11.4s)
{
  "id": "chatcmpl-9c556f212d3d4fbfa42d3896e2d0baa2",
  "created": 1775943054,
  "model": "qwen3-coder-480b-a35b",
  "object": "chat.completion",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Вот пример функции сортировки пузырьком на Python:\n\n```python\ndef bubble_sort(arr):\n    \"\"\"\n    Сортировка пузырьком.\n    \n    Аргументы:\n        arr (list): Список для сортировки.\n        \n    Возвращает:\n        list: Отсортированный список.\n    \"\"\"\n    n = len(arr)\n    # Проходим по всем элементам списка\n    for i in r



## 4. Chat — Long Context (qwen2.5-72b-instruct)

In [66]:
r, t = timed_post(f"{BASE}/v1/chat/completions", json={
    "model": "qwen2.5-72b-instruct",
    "messages": [{"role": "user", "content": "Объясни трансформерные архитектуры кратко"}],
    "stream": False, "user": USER_ID
})
ok("Chat / Long Context (qwen2.5-72b-instruct)", r, t)

✅ [200] Chat / Long Context (qwen2.5-72b-instruct) (12.9s)
{
  "id": "chatcmpl-e3335a0199934781a06bf518e7001e95",
  "created": 1775943067,
  "model": "qwen2.5-72b-instruct",
  "object": "chat.completion",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Трансформеры - это тип архитектуры искусственных нейронных сетей, использованный в обучении с подкреплением и машинном обучении, особенно важный в областях обработки естественного языка и компьютерного зрения. \n\nОни были впервые представлены в статье \"Attention Is All You Need\" в 2017 году. \"Просмотр\" (или Attention) - это механизм, кото



## 5. Chat — Auto Router (model=auto, роутинг через MWS)

In [67]:
r, t = timed_post(f"{BASE}/v1/chat/completions", json={
    "model": "auto",
    "messages": [{"role": "user", "content": "Напиши алгоритм бинарного поиска"}],
    "stream": False, "user": USER_ID
})
ok("Chat / Auto Router → код", r, t)

✅ [200] Chat / Auto Router → код (21.0s)
{
  "id": "chatcmpl-f8d1e532ba0e46079b115e33c170d8a8",
  "created": 1775943078,
  "model": "qwen3-coder-480b-a35b",
  "object": "chat.completion",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Вот алгоритм бинарного поиска на Python:\n\n```python\ndef binary_search(arr, target):\n    \"\"\"\n    Бинарный поиск в отсортированном массиве\n    \n    Аргументы:\n    arr - отсортированный список элементов\n    target - искомый элемент\n    \n    Возвращает:\n    индекс элемента, если найден, иначе -1\n    \"\"\"\n    left = 0\n    right



## 6. Chat — Streaming (SSE)

In [68]:
print("🔄 Streaming:")
t0 = time.time()
chunks = []
with httpx.stream("POST", f"{BASE}/v1/chat/completions", json={
    "model": "mws-gpt-alpha",
    "messages": [{"role": "user", "content": "Расскажи короткий анекдот"}],
    "stream": True, "user": USER_ID
}, timeout=TIMEOUT) as resp:
    print(f"Status: {resp.status_code}")
    for line in resp.iter_lines():
        if line.startswith("data: ") and line != "data: [DONE]":
            try:
                delta = json.loads(line[6:])["choices"][0]["delta"].get("content", "")
                if delta: chunks.append(delta); print(delta, end="", flush=True)
            except: pass
t = time.time() - t0
print(f"\n\n✅ chunks: {len(chunks)}, time: {t:.1f}s")

🔄 Streaming:
Status: 200
Почему компьютер пошел в психолога?

Потому что у него была ошибка в программе! (смеется)

✅ chunks: 31, time: 2.1s


## 7. Embeddings (bge-m3)

In [69]:
r, t = timed_post(f"{BASE}/v1/embeddings", json={"model": "bge-m3", "input": "Тестовый текст"})
ok("Embeddings (bge-m3)", r, t)

✅ [200] Embeddings (bge-m3) (0.7s)
{
  "model": "/models/bge-m3",
  "data": [
    {
      "embedding": [
        -0.038339663,
        0.04211078,
        -0.0064238342,
        -0.0049958024,
        0.0042633,
        -0.040040363,
        0.010305864,
        0.025565937,
        -0.0187539,
        0.0048525366,
        -0.00010217767,
        -0.0018277889,
        -0.022552742,
        -0.02501136,
        -0.00080413464,
        -0.030464688,
        -0.0029623583,
        -0.00887321,
        -0.036934737,
        -0.019909265,
        -0.038413607,
        -0.0103890505,
        0.024900446,
        0.034993723,
      



## 8. List Models

In [70]:
r, t = timed_get(f"{BASE}/v1/models")
ok("List Models", r, t)

✅ [200] List Models (0.4s)
{
  "data": [
    {
      "id": "deepseek-r1-distill-qwen-32b",
      "object": "model",
      "created": 1677610602,
      "owned_by": "openai"
    },
    {
      "id": "gpt-oss-20b",
      "object": "model",
      "created": 1677610602,
      "owned_by": "openai"
    },
    {
      "id": "cotype-pro-vl-32b",
      "object": "model",
      "created": 1677610602,
      "owned_by": "openai"
    },
    {
      "id": "llama-3.1-8b-instruct",
      "object": "model",
      "created": 1677610602,
      "owned_by": "openai"
    },
    {
      "id": "QwQ-32B",
      "object": "model",
      "created"



## 9. Deep Research (SSE)

In [71]:
print("🔄 Deep Research:")
t0 = time.time()
with httpx.stream("POST", f"{BASE}/v1/research", json={"query": "Квантовые вычисления", "user_id": USER_ID}, timeout=TIMEOUT) as resp:
    print(f"Status: {resp.status_code}")
    for line in resp.iter_lines():
        if line: print(line[:120])
t = time.time() - t0
print(f"\n✅ Research done ({t:.1f}s)")

🔄 Deep Research:
Status: 200
event: progress
data: {"step": 1, "message": "Генерирую подзапросы…"}
event: progress
data: {"step": 2, "sub_queries": ["Квантовые вычисления"]}
event: progress
data: {"step": 3, "message": "Ищу и парсю страницы…"}
event: progress
data: {"step": 4, "pages_fetched": 3}
event: progress
data: {"step": 5, "message": "Синтезирую ответ…"}
event: done
data: {"answer": "На основе предоставленных источников, квантовые вычисления представляют собой область исследования и и

✅ Research done (24.6s)


## 10. Web Search

In [72]:
r, t = timed_post(f"{BASE}/v1/tools/web-search", json={"query": "FastAPI Python 2025", "max_results": 3})
ok("Web Search", r, t)

✅ [200] Web Search (1.6s)
{
  "results": [
    {
      "title": "fastapi · PyPI",
      "url": "https://pypi.org/project/fastapi/",
      "snippet": "FastAPIis a modern, fast (high-performance), web framework for building APIs withPythonbased on standardPythontype hints. The key features are: Fast: Very high performance, on par with NodeJS and Go (thanks to Starlette and Pydantic). One of the fastestPythonframeworks available. Fast to code: Increase the speed to develop features by about 200% to 300%. * Fewer bugs: Reduce about 40% ..."
    },
    {
      "title": "FastAPI in 2025: Why Developers Are Choosing This Mode



## 11. Web Parse

In [87]:
r, t = timed_post(f"{BASE}/v1/tools/web-parse", json={"url": "https://ru.wikipedia.org/wiki/%D0%92%D0%B8%D0%BA%D0%B8%D0%BF%D0%B5%D0%B4%D0%B8%D1%8F"})
ok("Web Parse", r, t)

✅ [200] Web Parse (1.1s)
{
  "url": "https://ru.wikipedia.org/wiki/%D0%92%D0%B8%D0%BA%D0%B8%D0%BF%D0%B5%D0%B4%D0%B8%D1%8F",
  "text": "Википедия — Википедия\nПерейти к содержанию\nЭта страница защищена от редактирования (частичная защита (до автоподтверждённых)).\nМатериал из Википедии — свободной энциклопедии\nСтабильная версия\n, проверенная 30 марта 2026.\nУ этого термина существуют и другие значения, см.\nВикипедия (значения)\n.\nЭта статья — о многоязычном веб-сайте, являющемся одним из проектов\nWikimedia Foundation\n. О разделах энциклопедии на разных языках см.\nЯзыковые разделы Википедии\n; о данном (русскояз



## 12. File Upload — TXT

In [74]:
with open('/tmp/test_doc.txt', 'rb') as f:
    r, t = timed_post(f"{BASE}/v1/files/upload",
        files={"file": ("test_doc.txt", f, "text/plain")},
        data={"user_id": USER_ID})
ok("File Upload (TXT)", r, t)

✅ [200] File Upload (TXT) (0.6s)
{
  "file_id": "5e383d2b-55f7-40b9-b80c-9188ecd29287",
  "chunks": 1
}



## 13. File Upload — PDF

In [75]:
with open('/tmp/test_doc.pdf', 'rb') as f:
    r, t = timed_post(f"{BASE}/v1/files/upload",
        files={"file": ("test_doc.pdf", f, "application/pdf")},
        data={"user_id": USER_ID})
ok("File Upload (PDF)", r, t)

✅ [200] File Upload (PDF) (0.8s)
{
  "file_id": "391cbe0d-b416-4800-bbfb-54e146db2245",
  "chunks": 1
}



## 14. File Upload — DOCX

In [76]:
with open('/tmp/test_doc.docx', 'rb') as f:
    r, t = timed_post(f"{BASE}/v1/files/upload",
        files={"file": ("test_doc.docx", f, "application/vnd.openxmlformats-officedocument.wordprocessingml.document")},
        data={"user_id": USER_ID})
ok("File Upload (DOCX)", r, t)

✅ [200] File Upload (DOCX) (0.9s)
{
  "file_id": "e30df61c-e49e-4311-b283-ec3c5e3528ac",
  "chunks": 1
}



## 15. File QA (chat с файлом)

In [77]:
r, t = timed_post(f"{BASE}/v1/chat/completions", json={
    "model": "auto",
    "messages": [{"role": "user", "content": "Что написано в моём документе?"}],
    "stream": False, "user": USER_ID,
    "attachments": [{"name": "test_doc.txt", "mime": "text/plain"}]
})
ok("Chat / File QA", r, t)

✅ [200] Chat / File QA (3.1s)
{
  "id": "chatcmpl-a5ac2238773c24b9",
  "created": 1775943132,
  "model": "mws-gpt-alpha",
  "object": "chat.completion",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Я не имею возможности видеть содержимое вашего документа, поскольку я — это большая языковая модель, основанная на архитектуре GPT, и у меня нет прямого доступа к вашему компьютеру или файлам.\n\nЕсли вы хотите узнать содержимое своего документа, вы можете:\n\n1. Отсканировать или фотографировать документ и отправить его мне в виде изображения.\n2. Вставить содержим



## 16. Memory — GET

In [78]:
r, t = timed_get(f"{BASE}/v1/memory/{USER_ID}")
ok("Memory GET", r, t)

✅ [200] Memory GET (0.0s)
{
  "user_id": "test_user",
  "memories": []
}



## 17. Memory — Search

In [79]:
r, t = timed_post(f"{BASE}/v1/memory/{USER_ID}/search", json={"query": "Python", "top_k": 5})
ok("Memory Search", r, t)

✅ [200] Memory Search (0.5s)
{
  "user_id": "test_user",
  "results": []
}



## 18. Image Generation (qwen-image через MWS)

In [80]:
r, t = timed_post(f"{BASE}/v1/image/generate", json={
    "prompt": "красивый закат над морем",
    "width": 512, "height": 512
})
ok("Image Generate (qwen-image → MWS)", r, t)

✅ [200] Image Generate (qwen-image → MWS) (0.5s)
{
  "image_b64": null,
  "mime": null,
  "fallback": true,
  "description": "Сервис генерации изображений временно недоступен. Вот текстовое описание:\n\n🎨 красивый закат над морем\n\nРазмер: 512×512px, шагов: 20."
}



## 19. VLM Analyze (multipart — файл + вопрос)

In [81]:
try:
    img = httpx.get('https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png',
        follow_redirects=True, timeout=10, verify=False).content
except:
    img = bytes.fromhex('89504e470d0a1a0a0000000d49484452000000010000000108060000001f15c489'
                        '0000000a49444154789c626000000002000198e195280000000049454e44ae426082')

r, t = timed_post(f"{BASE}/v1/vlm/analyze",
    files={"image": ("test.png", img, "image/png")},
    data={"question": "Что изображено?"})
ok("VLM Analyze", r, t)

✅ [200] VLM Analyze (0.0s)
{
  "answer": null,
  "fallback": true,
  "message": "Сервис анализа изображений (VLM) временно недоступен. Пожалуйста, попробуйте позже или опишите изображение текстом."
}



## 20. PPTX Generation

In [82]:
t0 = time.time()
r = httpx.post(f"{BASE}/v1/tools/generate-pptx", json={
    "topic": "Искусственный интеллект",
    "slide_count": 5, "language": "ru"
}, timeout=TIMEOUT)
t = time.time() - t0
if r.status_code == 200 and 'application' in r.headers.get('content-type', ''):
    out = Path('/tmp/test_output.pptx')
    out.write_bytes(r.content)
    print(f"✅ [200] PPTX saved → {out} ({len(r.content):,} bytes) ({t:.1f}s)")
else:
    ok("PPTX Generate", r, t)

❌ [500] PPTX Generate (3.3s)
Internal Server Error



## 21. Voice Message (whisper-turbo-local через MWS)

In [83]:
with open('/tmp/test_audio.wav', 'rb') as f:
    t0 = time.time()
    r = httpx.post(f"{BASE}/v1/voice/message",
        files={"audio": ("test_audio.wav", f, "audio/wav")},
        data={"user_id": USER_ID},
        timeout=TIMEOUT)
    t = time.time() - t0
if r.status_code == 200:
    print(f"✅ [200] Voice pipeline OK ({t:.1f}s)")
    print(f"  Transcript: {r.headers.get('X-Transcript', '?')}")
    print(f"  Answer: {r.headers.get('X-Answer', '?')[:200]}")
    print(f"  Audio: {len(r.content):,} bytes")
elif r.status_code == 503:
    print(f"⚠️  [503] ASR недоступен ({t:.1f}s) — whisper-turbo-local и media-service оба не ответили")
else:
    ok("Voice Message", r, t)

❌ [500] Voice Message (2.9s)
Internal Server Error



## 22. Router Detection — все типы задач (через MWS API)

In [84]:
cases = [
    ("Привет, как дела?",                          "text",          "mws-gpt-alpha"),
    ("Напиши функцию сортировки на Python",         "code",          "qwen3-coder-480b-a35b"),
    ("Исследуй тему квантовых вычислений",          "deep_research", "qwen2.5-72b-instruct"),
    ("Зайди на https://example.com и расскажи",     "web_parse",     "mws-gpt-alpha"),
    ("Нарисуй закат над морем",                     "image_gen",     "qwen-image"),
    ("Глубокий анализ блокчейна",                  "deep_research", "qwen2.5-72b-instruct"),
    ("Реализуй алгоритм Дейкстры на Go",           "code",          "qwen3-coder-480b-a35b"),
]
print(f"{'Ожид. тип':<18} {'Ожид. модель':<30} {'Время':>7} Статус | Сообщение")
print("-" * 95)
for msg, exp_task, exp_model in cases:
    try:
        t0 = time.time()
        r = httpx.post(f"{BASE}/v1/chat/completions", json={
            "model": "auto",
            "messages": [{"role": "user", "content": msg}],
            "stream": False, "user": USER_ID
        }, timeout=TIMEOUT)
        t = time.time() - t0
        icon = '✅' if r.status_code < 400 else '❌'
        print(f"{exp_task:<18} {exp_model:<30} {t:>6.1f}s {icon} [{r.status_code}] | {msg[:40]}")
    except Exception as e:
        print(f"{exp_task:<18} {exp_model:<30}         ❌ ERR: {e}")

Ожид. тип          Ожид. модель                     Время Статус | Сообщение
-----------------------------------------------------------------------------------------------
text               mws-gpt-alpha                     3.1s ✅ [200] | Привет, как дела?
code               qwen3-coder-480b-a35b            20.9s ✅ [200] | Напиши функцию сортировки на Python
deep_research      qwen2.5-72b-instruct             27.7s ✅ [200] | Исследуй тему квантовых вычислений
web_parse          mws-gpt-alpha                     4.6s ✅ [200] | Зайди на https://example.com и расскажи
image_gen          qwen-image                       12.6s ❌ [404] | Нарисуй закат над морем
deep_research      qwen2.5-72b-instruct             25.7s ✅ [200] | Глубокий анализ блокчейна
code               qwen3-coder-480b-a35b            28.2s ✅ [200] | Реализуй алгоритм Дейкстры на Go


## 23. Роутер — детерминированные правила (без API)

In [85]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.environ.setdefault('MWS_API_KEY', 'test')
os.environ.setdefault('DATABASE_URL', 'postgresql+asyncpg://u:p@localhost/db')
os.environ.setdefault('REDIS_URL', 'redis://localhost:6379/0')

from app.config import get_settings
from app.services.router_client import RouterClient

router = RouterClient(get_settings())

cases = [
    ('Привет, как дела?',                        [],                                          None),
    ('Напиши функцию сортировки на Python',       [],                                          'code'),
    ('Реализуй алгоритм Дейкстры',               [],                                          'code'),
    ('Исследуй тему квантовых вычислений',        [],                                          'deep_research'),
    ('Глубокий анализ блокчейна',                [],                                          'deep_research'),
    ('Зайди на https://example.com',              [],                                          'web_parse'),
    ('',                                          [{'name':'voice.mp3','mime':'audio/mpeg'}],  'asr'),
    ('что на фото?',                              [{'name':'img.jpg','mime':'image/jpeg'}],    'vlm'),
    ('разбери документ',                          [{'name':'doc.pdf','mime':'application/pdf'}],'file_qa'),
    ('',                                          [{'name':'report.docx','mime':'application/vnd.openxmlformats-officedocument.wordprocessingml.document'}], 'file_qa'),
]

print(f"{'Сообщение':<45} {'Ожидается':<15} {'Результат':<15} OK?")
print('-' * 80)
passed = 0
for msg, att, expected in cases:
    result = router._deterministic(msg, att)
    got = result.task_type if result else None
    match = got == expected
    if match: passed += 1
    label = msg or att[0]['name']
    print(f"{label[:44]:<45} {str(expected):<15} {str(got):<15} {'✅' if match else '❌'}")
print(f'\nДетерминированный роутер: {passed}/{len(cases)} ✅')

Сообщение                                     Ожидается       Результат       OK?
--------------------------------------------------------------------------------
Привет, как дела?                             None            None            ✅
Напиши функцию сортировки на Python           code            code            ✅
Реализуй алгоритм Дейкстры                    code            code            ✅
Исследуй тему квантовых вычислений            deep_research   deep_research   ✅
Глубокий анализ блокчейна                     deep_research   deep_research   ✅
Зайди на https://example.com                  web_parse       web_parse       ✅
voice.mp3                                     asr             asr             ✅
что на фото?                                  vlm             vlm             ✅
разбери документ                              file_qa         file_qa         ✅
report.docx                                   file_qa         file_qa         ✅

Детерминированный роутер: 10/10 ✅


## 24. Summary — All Endpoints

In [86]:
endpoints = [
    ("GET",  "/v1/health",                    None),
    ("POST", "/v1/chat/completions",          {"model":"mws-gpt-alpha","messages":[{"role":"user","content":"hi"}],"stream":False}),
    ("POST", "/v1/embeddings",                {"model":"bge-m3","input":"test"}),
    ("GET",  "/v1/models",                    None),
    ("POST", "/v1/tools/web-search",          {"query":"test","max_results":1}),
    ("POST", "/v1/tools/web-parse",           {"url":"https://example.com"}),
    ("POST", "/v1/tools/generate-pptx",       {"topic":"AI","slide_count":3,"language":"ru"}),
    ("GET",  f"/v1/memory/{USER_ID}",         None),
    ("POST", f"/v1/memory/{USER_ID}/search",  {"query":"test","top_k":3}),
    ("POST", "/v1/image/generate",            {"prompt":"cat"}),
]
print(f"{'Method':<6} {'Endpoint':<40} {'Status':>6} {'Time':>7}  Result")
print("-" * 70)
passed = 0
for method, path, body in endpoints:
    try:
        t0 = time.time()
        r = httpx.get(f"{BASE}{path}", timeout=TIMEOUT) if method == "GET" else httpx.post(f"{BASE}{path}", json=body, timeout=TIMEOUT)
        t = time.time() - t0
        icon = "✅" if r.status_code < 400 else "⚠️ "
        if r.status_code < 400: passed += 1
        print(f"{method:<6} {path:<40} {r.status_code:>6} {t:>6.1f}s  {icon}")
    except Exception as e:
        print(f"{method:<6} {path:<40} {'ERR':>6}          ❌ {str(e)[:30]}")
print(f"\n{'='*70}\nИтого: {passed}/{len(endpoints)} OK")

Method Endpoint                                 Status    Time  Result
----------------------------------------------------------------------
GET    /v1/health                                  200    0.6s  ✅
POST   /v1/chat/completions                        200    2.4s  ✅
POST   /v1/embeddings                              200    0.5s  ✅
GET    /v1/models                                  200    0.4s  ✅
POST   /v1/tools/web-search                        200    0.7s  ✅
POST   /v1/tools/web-parse                         200    0.3s  ✅
POST   /v1/tools/generate-pptx                     200    1.8s  ✅
GET    /v1/memory/test_user                        200    0.0s  ✅
POST   /v1/memory/test_user/search                 200    0.7s  ✅
POST   /v1/image/generate                          200    0.5s  ✅

Итого: 10/10 OK
